# 3D U-Field

This notebook is a compact scalar-only 3D lift of the precursor 2D U-Field implementation.

## Helper primitives

In [ ]:

from __future__ import annotations

from pathlib import Path
from typing import Any
import itertools
import math

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation, colors


def clamp(x: np.ndarray | float, lo: float, hi: float) -> np.ndarray:
    return np.clip(x, lo, hi)


def logistic_phi(rho: np.ndarray, alpha: float, rho_c: float) -> np.ndarray:
    """
    logistic activation response:
        Phi(rho) = 1 / (1 + exp(-alpha * (rho - rho_c)))
    """
    x = clamp(alpha * (rho - rho_c), -60.0, 60.0)
    return 1.0 / (1.0 + np.exp(-x))


def recursive_rhythm_depth(phi: np.ndarray, gamma: np.ndarray) -> np.ndarray:
    """
    R = Phi * Gamma
    """
    return clamp(phi * gamma, 0.0, 1.0)


def shift_scalar(field: np.ndarray, axis: int, offset: int, fill_value: float = 0.0) -> np.ndarray:
    """
    Shift a 3D scalar field with open boundaries (Dirichlet zero padding).

    """
    if field.ndim != 3:
        raise ValueError("shift_scalar expects a 3D scalar field.")
    if axis not in (0, 1, 2):
        raise ValueError("axis must be 0, 1, or 2.")
    if offset == 0:
        return field.copy()

    out = np.full_like(field, fill_value)
    n = field.shape[axis]

    src = [slice(None)] * 3
    dst = [slice(None)] * 3

    if offset > 0:
        src[axis] = slice(0, n - offset)
        dst[axis] = slice(offset, n)
    else:
        k = -offset
        src[axis] = slice(k, n)
        dst[axis] = slice(0, n - k)

    out[tuple(dst)] = field[tuple(src)]
    return out


def build_neighbor_offsets_26(weight_mode: str = "inverse_distance") -> tuple[list[tuple[int, int, int]], np.ndarray]:
    """
    26-neighbor Moore-family stencil for the 3D scalar implementation.
    """
    offsets: list[tuple[int, int, int]] = []
    weights: list[float] = []

    for dz, dy, dx in itertools.product((-1, 0, 1), repeat=3):
        if dz == dy == dx == 0:
            continue

        dist = math.sqrt(dx * dx + dy * dy + dz * dz)
        offsets.append((dz, dy, dx))

        if weight_mode == "uniform":
            w = 1.0
        elif weight_mode == "inverse_distance":
            w = 1.0 / dist
        elif weight_mode == "inverse_distance_sq":
            w = 1.0 / (dist * dist)
        else:
            raise ValueError("Unsupported weight_mode.")

        weights.append(w)

    weight_array = np.asarray(weights, dtype=np.float64)
    weight_array /= np.sum(weight_array)
    return offsets, weight_array


def local_neighbor_mean_scalar(
    field: np.ndarray,
    offsets: list[tuple[int, int, int]],
    weights: np.ndarray,
) -> np.ndarray:
    """
    Matched isotropic local mean on the 3D Moore family.
    """
    out = np.zeros_like(field)
    for (dz, dy, dx), w in zip(offsets, weights):
        shifted = shift_scalar(field, axis=0, offset=dz)
        shifted = shift_scalar(shifted, axis=1, offset=dy)
        shifted = shift_scalar(shifted, axis=2, offset=dx)
        out += w * shifted
    return out


def grad_scalar(field: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Matched isotropic 3D Sobel-like gradient with open boundaries.

    Returns:
        d/dx, d/dy, d/dz
    """
    smooth = {-1: 1.0, 0: 2.0, 1: 1.0}

    gx = np.zeros_like(field)
    gy = np.zeros_like(field)
    gz = np.zeros_like(field)

    for dz, dy, dx in itertools.product((-1, 0, 1), repeat=3):
        shifted = shift_scalar(shift_scalar(shift_scalar(field, 0, dz), 1, dy), 2, dx)

        if dx != 0:
            gx += smooth[dy] * smooth[dz] * (1.0 if dx > 0 else -1.0) * shifted
        if dy != 0:
            gy += smooth[dx] * smooth[dz] * (1.0 if dy > 0 else -1.0) * shifted
        if dz != 0:
            gz += smooth[dx] * smooth[dy] * (1.0 if dz > 0 else -1.0) * shifted

    return gx / 32.0, gy / 32.0, gz / 32.0


def divergence_vector(vx: np.ndarray, vy: np.ndarray, vz: np.ndarray) -> np.ndarray:
    """
    Matched isotropic divergence built from the same discrete geometry as grad_scalar().
    """
    dvx_dx, _, _ = grad_scalar(vx)
    _, dvy_dy, _ = grad_scalar(vy)
    _, _, dvz_dz = grad_scalar(vz)
    return dvx_dx + dvy_dy + dvz_dz


def history_pairwise_similarity(history: np.ndarray, eps: float) -> np.ndarray:
    if history.shape[0] < 2:
        return np.zeros_like(history[-1])

    a = history[:-1]
    b = history[1:]

    numerator = np.sum(a * b, axis=0)
    denom = np.sqrt(np.sum(a * a, axis=0)) * np.sqrt(np.sum(b * b, axis=0)) + eps
    return clamp(numerator / denom, -1.0, 1.0)


def history_sign_flip_regularity(history: np.ndarray) -> np.ndarray:
    if history.shape[0] < 2:
        return np.zeros_like(history[-1])

    a = history[:-1]
    b = history[1:]

    sa = np.sign(a)
    sb = np.sign(b)

    active_pairs = (np.abs(a) + np.abs(b)) > 0.0
    signed_pairs = (sa != 0) & (sb != 0) & active_pairs
    flip_pairs = (sa != sb) & signed_pairs

    valid_pairs = np.sum(signed_pairs, axis=0)
    flip_count = np.sum(flip_pairs, axis=0)

    out = np.zeros_like(history[-1])
    np.divide(flip_count, valid_pairs, out=out, where=valid_pairs > 0)
    return out


def history_magnitude_consistency(history: np.ndarray, eps: float) -> np.ndarray:
    mags = np.abs(history)
    mean_mag = np.mean(mags, axis=0)
    std_mag = np.std(mags, axis=0)
    return clamp(1.0 - std_mag / (mean_mag + eps), 0.0, 1.0)


def history_activity_gate(history: np.ndarray, eps: float) -> np.ndarray:
    mags = np.abs(history)
    max_mag = np.max(mags, axis=0)
    mean_mag = np.mean(mags, axis=0)

    out = np.zeros_like(history[-1])
    np.divide(mean_mag, max_mag + eps, out=out, where=max_mag > 0.0)
    return clamp(out, 0.0, 1.0)


## Compact scalar-only 3D execution path

In [ ]:

from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict


@dataclass
class ScalarConfig:
    """
    Compact scalar-only 3D lift of the 2D U-field.

    The intent is strict continuity with the 2D scenario:
    - scalar carrier field u
    - logistic activation Phi(rho)
    - Gamma proxy from history terms C_rec, M_mag, A_hist
    - R = Phi * Gamma
    - signed transport branch via chi * R
    - flux = div(Phi * grad(R_signed))
    - decoherence from activation-gradient structure
    """
    nz: int = 75
    ny: int = 75
    nx: int = 75

    # Number of evolved steps after the seeded state.
    steps: int = 500

    # Rolling history window length.
    w: int = 6

    alpha: float = 10.0
    rho_c: float = 0.5

    u0: float = 1.0e-3
    seed_sign: int = 1

    eps_rho: float = 1.0e-12
    eps_c: float = 1.0e-12
    eps_m: float = 1.0e-12
    eps_x: float = 1.0e-12

    # 2D implementation motion-law weights carried directly into 3D.
    lambda_delta: float = 1.00
    lambda_flux: float = 0.50
    lambda_decoherence: float = 0.25

    capture_every: int = 1
    summary_every: int = 10

    # Point-cloud rendering / export
    max_points: int = 20000  # (7000, 20000)
    cmap: str = "coolwarm"
    export_display_percentile: float = 99.5
    fps: int = 12
    interval_ms: int = 80
    output_mp4: str = "3d_u_field.mp4"
    title: str = "3D U-Field"

    def validate(self) -> None:
        for name, n in (("nz", self.nz), ("ny", self.ny), ("nx", self.nx)):
            if n <= 0:
                raise ValueError(f"{name} must be positive.")
            if n % 2 == 0:
                raise ValueError(f"{name} must be odd so the seed sits exactly at the origin.")

        if self.steps < 0:
            raise ValueError("steps must be non-negative.")
        if self.w < 3:
            raise ValueError("w must be >= 3.")
        if self.alpha <= 0.0:
            raise ValueError("alpha must be > 0.")
        if self.u0 <= 0.0:
            raise ValueError("u0 must be > 0.")
        if self.seed_sign not in (-1, 1):
            raise ValueError("seed_sign must be +1 or -1.")

        if self.lambda_delta < 0.0 or self.lambda_flux < 0.0 or self.lambda_decoherence < 0.0:
            raise ValueError("motion-law weights must be non-negative.")

        if self.capture_every <= 0 or self.summary_every <= 0:
            raise ValueError("capture_every and summary_every must be >= 1.")

        if not (0.0 < self.export_display_percentile <= 100.0):
            raise ValueError("export_display_percentile must be in (0, 100].")
        if self.max_points < 1:
            raise ValueError("max_points must be >= 1.")


class Scalar3DSim:
    def __init__(self, config: ScalarConfig):
        self.cfg = config
        self.cfg.validate()

        self.nz, self.ny, self.nx = self.cfg.nz, self.cfg.ny, self.cfg.nx
        self.cz, self.cy, self.cx = self.nz // 2, self.ny // 2, self.nx // 2

        self.offsets, self.weights = build_neighbor_offsets_26("inverse_distance")

        # Core scalar state and observables
        self.u = np.zeros((self.nz, self.ny, self.nx), dtype=np.float64)
        self.u_mean = np.zeros_like(self.u)
        self.delta = np.zeros_like(self.u)

        self.rho = np.zeros_like(self.u)
        self.phi = np.zeros_like(self.u)

        self.C_rec = np.zeros_like(self.u)
        self.F_flip = np.zeros_like(self.u)
        self.M_mag = np.zeros_like(self.u)
        self.A_hist = np.zeros_like(self.u)

        self.Gamma = np.zeros_like(self.u)
        self.R = np.zeros_like(self.u)

        self.chi = np.zeros_like(self.u)
        self.R_signed = np.zeros_like(self.u)

        self.grad_phi_x = np.zeros_like(self.u)
        self.grad_phi_y = np.zeros_like(self.u)
        self.grad_phi_z = np.zeros_like(self.u)

        self.grad_R_x = np.zeros_like(self.u)
        self.grad_R_y = np.zeros_like(self.u)
        self.grad_R_z = np.zeros_like(self.u)

        self.grad_phi_mag = np.zeros_like(self.u)
        self.grad_R_mag = np.zeros_like(self.u)

        self.Ix = np.zeros_like(self.u)
        self.Iy = np.zeros_like(self.u)
        self.Iz = np.zeros_like(self.u)
        self.I_mag = np.zeros_like(self.u)

        self.Gamma_D = np.zeros_like(self.u)
        self.flux_proj = np.zeros_like(self.u)
        self.decoherence_term = np.zeros_like(self.u)

        self.history_buffer = np.zeros((self.cfg.w, self.nz, self.ny, self.nx), dtype=np.float64)

        self.u_frames: list[np.ndarray] = []
        self.summaries: list[dict[str, Any]] = []

    def neighbor_mean(self, field: np.ndarray) -> np.ndarray:
        return local_neighbor_mean_scalar(field, self.offsets, self.weights)


    @staticmethod
    def masked_mean(field: np.ndarray, mask: np.ndarray) -> float:
        if np.any(mask):
            return float(np.mean(field[mask]))
        return 0.0

    def compute_interface_mask(self, field: np.ndarray) -> np.ndarray:
        """
        Boolean mask for signed voxels that touch at least one opposite-sign
        26-neighbor voxel. Zero-valued voxels are not counted as interface voxels.
        """
        s = np.sign(field).astype(np.int8)
        signed_mask = s != 0

        p = np.pad(s, 1, mode="constant", constant_values=0)
        center = p[1:-1, 1:-1, 1:-1]

        opposite_touch = np.zeros_like(signed_mask, dtype=bool)
        for dz in (-1, 0, 1):
            for dy in (-1, 0, 1):
                for dx in (-1, 0, 1):
                    if dz == 0 and dy == 0 and dx == 0:
                        continue
                    neighbor = p[
                        1 + dz : self.nz + 1 + dz,
                        1 + dy : self.ny + 1 + dy,
                        1 + dx : self.nx + 1 + dx,
                    ]
                    opposite_touch |= (center * neighbor < 0)

        return signed_mask & opposite_touch

    def compute_high_contrast_mask(self) -> np.ndarray:
        """
        Diagnostic-only high-contrast mask.
        Uses the native activation threshold rho_c, so phi >= 0.5 is equivalent.
        """
        return self.rho >= self.cfg.rho_c


    def compute_observables_for_current_state(self, field: np.ndarray) -> None:
        # Local relational geometry
        self.u_mean = self.neighbor_mean(field)
        self.delta = self.u_mean - field

        denom = self.cfg.eps_rho + np.abs(field) + np.abs(self.u_mean)
        self.rho = np.abs(self.delta) / denom
        self.phi = logistic_phi(self.rho, self.cfg.alpha, self.cfg.rho_c)

        # History-derived diagnostics / proxies
        self.C_rec = history_pairwise_similarity(self.history_buffer, self.cfg.eps_c)
        self.F_flip = history_sign_flip_regularity(self.history_buffer)
        self.M_mag = history_magnitude_consistency(self.history_buffer, self.cfg.eps_m)
        self.A_hist = history_activity_gate(self.history_buffer, self.cfg.eps_m)

        self.Gamma = clamp(self.A_hist * self.M_mag * np.abs(self.C_rec), 0.0, 1.0)
        self.R = recursive_rhythm_depth(self.phi, self.Gamma)

        # Smooth odd orientation carrier for sign-neutral recursive transport
        orient_denom = self.cfg.eps_x + np.abs(field) + np.abs(self.u_mean)
        self.chi = np.divide(field, orient_denom, out=np.zeros_like(field), where=orient_denom > 0.0)
        self.chi = clamp(self.chi, -1.0, 1.0)
        self.R_signed = self.chi * self.R

        # Matched gradient / divergence geometry
        self.grad_phi_x, self.grad_phi_y, self.grad_phi_z = grad_scalar(self.phi)
        self.grad_R_x, self.grad_R_y, self.grad_R_z = grad_scalar(self.R_signed)

        self.grad_phi_mag = np.sqrt(self.grad_phi_x**2 + self.grad_phi_y**2 + self.grad_phi_z**2)
        self.grad_R_mag = np.sqrt(self.grad_R_x**2 + self.grad_R_y**2 + self.grad_R_z**2)

        self.Ix = self.phi * self.grad_R_x
        self.Iy = self.phi * self.grad_R_y
        self.Iz = self.phi * self.grad_R_z
        self.I_mag = np.sqrt(self.Ix**2 + self.Iy**2 + self.Iz**2)

        scale = self.grad_phi_mag + np.abs(self.R)
        self.Gamma_D = np.divide(
            self.grad_phi_mag,
            self.cfg.eps_x + scale,
            out=np.zeros_like(self.u),
            where=(self.cfg.eps_x + scale) > 0.0,
        )
        self.Gamma_D = clamp(self.Gamma_D, 0.0, 1.0)

    def compute_summary(self, step_index: int) -> dict[str, Any]:
        interface_mask = self.compute_interface_mask(self.u)
        signed_mask = np.sign(self.u).astype(np.int8) != 0
        off_interface_signed_mask = signed_mask & (~interface_mask)

        high_contrast_mask = self.compute_high_contrast_mask()
        high_contrast_signed_mask = high_contrast_mask & signed_mask
        high_contrast_interface_mask = high_contrast_signed_mask & interface_mask
        high_contrast_off_interface_mask = high_contrast_signed_mask & (~interface_mask)

        interface_occupancy_ratio = float(np.mean(interface_mask))

        mean_rho_interface = self.masked_mean(self.rho, interface_mask)
        mean_rho_off_interface = self.masked_mean(self.rho, off_interface_signed_mask)
        rho_interface_enrichment = mean_rho_interface / (mean_rho_off_interface + self.cfg.eps_rho)

        mean_F_flip_interface = self.masked_mean(self.F_flip, interface_mask)
        mean_F_flip_off_interface = self.masked_mean(self.F_flip, off_interface_signed_mask)
        bulk_oscillation_ratio = mean_F_flip_off_interface / (mean_F_flip_interface + self.cfg.eps_x)

        rho_interface_share = float(
            np.sum(self.rho[interface_mask]) /
            (np.sum(self.rho[signed_mask]) + self.cfg.eps_rho)
        )

        high_contrast_count = float(np.count_nonzero(high_contrast_signed_mask))
        high_contrast_interface_fraction = float(
            np.count_nonzero(high_contrast_interface_mask) / (high_contrast_count + self.cfg.eps_x)
        )
        high_contrast_off_interface_fraction = float(
            np.count_nonzero(high_contrast_off_interface_mask) / (high_contrast_count + self.cfg.eps_x)
        )
        high_contrast_rho_off_interface_share = float(
            np.sum(self.rho[high_contrast_off_interface_mask]) /
            (np.sum(self.rho[high_contrast_signed_mask]) + self.cfg.eps_rho)
        )

        return {
            "step": step_index,
            "mean_abs_u": float(np.mean(np.abs(self.u))),
            "mean_rho": float(np.mean(self.rho)),
            "mean_phi": float(np.mean(self.phi)),
            "mean_Gamma": float(np.mean(self.Gamma)),
            "mean_R": float(np.mean(self.R)),
            "mean_Gamma_D": float(np.mean(self.Gamma_D)),
            "mean_grad_phi_mag": float(np.mean(self.grad_phi_mag)),
            "mean_grad_R_mag": float(np.mean(self.grad_R_mag)),
            "mean_I_mag": float(np.mean(self.I_mag)),
            "mean_abs_C_rec": float(np.mean(np.abs(self.C_rec))),
            "mean_F_flip": float(np.mean(self.F_flip)),
            "mean_M_mag": float(np.mean(self.M_mag)),
            "mean_A_hist": float(np.mean(self.A_hist)),

            # 3D interface-shell analogs of the 2D boundary diagnostics
            "interface_occupancy_ratio": interface_occupancy_ratio,
            "rho_interface_enrichment": float(rho_interface_enrichment),
            "bulk_oscillation_ratio": float(bulk_oscillation_ratio),
            "rho_interface_share": rho_interface_share,

            # Additional 3D-specific high-contrast localization diagnostics
            "high_contrast_interface_fraction": high_contrast_interface_fraction,
            "high_contrast_off_interface_fraction": high_contrast_off_interface_fraction,
            "high_contrast_rho_off_interface_share": high_contrast_rho_off_interface_share,

            "mean_flux_proj": float(np.mean(np.abs(self.flux_proj))),
            "mean_decoherence_term": float(np.mean(np.abs(self.decoherence_term))),
            "pos_fraction": float(np.mean(self.u > 0.0)),
            "neg_fraction": float(np.mean(self.u < 0.0)),
            "u_min": float(np.min(self.u)),
            "u_max": float(np.max(self.u)),
        }

    def _capture_frame(self, frame_index: int) -> None:
        if frame_index % self.cfg.capture_every != 0:
            return
        self.u_frames.append(self.u.copy())

    def _capture_summary(self, step_index: int) -> None:
        summary = self.compute_summary(step_index)
        self.summaries.append(summary)

        if step_index % self.cfg.summary_every == 0:
            print(
                f"step {summary['step']:4d} | "
                f"<|u|> {summary['mean_abs_u']:.3e} | "
                f"<rho> {summary['mean_rho']:.3e} | "
                f"<phi> {summary['mean_phi']:.3e} | "
                f"<Gamma> {summary['mean_Gamma']:.3e} | "
                f"<R> {summary['mean_R']:.3e} | "
                f"<Gamma_D> {summary['mean_Gamma_D']:.3e} | "
                f"<|grad_phi|> {summary['mean_grad_phi_mag']:.3e} | "
                f"<|grad_R|> {summary['mean_grad_R_mag']:.3e} | "
                f"<|I|> {summary['mean_I_mag']:.3e} | "
                f"<|C_rec|> {summary['mean_abs_C_rec']:.3e} | "
                f"<F_flip> {summary['mean_F_flip']:.3e} | "
                f"<M_mag> {summary['mean_M_mag']:.3e} | "
                f"<A_hist> {summary['mean_A_hist']:.3e} | "
                f"r_interface {summary['interface_occupancy_ratio']:.3e} | "
                f"E_rho_interface {summary['rho_interface_enrichment']:.3e} | "
                f"O_bulk {summary['bulk_oscillation_ratio']:.3e} | "
                f"S_rho_interface {summary['rho_interface_share']:.3e} | "
                f"r_high_if {summary['high_contrast_interface_fraction']:.3e} | "
                f"r_high_off_if {summary['high_contrast_off_interface_fraction']:.3e} | "
                f"S_rho_high_off_if {summary['high_contrast_rho_off_interface_share']:.3e} | "
                f"<|flux|> {summary['mean_flux_proj']:.3e} | "
                f"<|decohere|> {summary['mean_decoherence_term']:.3e} | "
                f"+frac {summary['pos_fraction']:.3e} | "
                f"-frac {summary['neg_fraction']:.3e} | "
                f"u_min {summary['u_min']:.3e} | "
                f"u_max {summary['u_max']:.3e}"
            )

    def initialize_quiescent_frame(self) -> None:
        """
        Display-only blank frame before the seeded 2D-style scenario begins.
        """
        self.u.fill(0.0)
        self.flux_proj.fill(0.0)
        self.decoherence_term.fill(0.0)
        self.history_buffer.fill(0.0)
        self._capture_frame(frame_index=0)

    def inject_seeded_state(self) -> None:
        """
        Seeded state corresponding to step 0 of the lifted 2D scenario.
        """
        self.u.fill(0.0)
        self.u[self.cz, self.cy, self.cx] = float(self.cfg.seed_sign) * self.cfg.u0

        self.history_buffer[0] = self.u.copy()
        self.history_buffer[1:] = 0.0

        self.compute_observables_for_current_state(self.u)
        self._capture_frame(frame_index=1)
        self._capture_summary(step_index=0)

    def step(self, step_index: int) -> None:
        # Same reduced scalar motion law as the 2D implementation, now on a 3D scalar field
        self.flux_proj = divergence_vector(self.Ix, self.Iy, self.Iz)

        delta_term = self.cfg.lambda_delta * self.phi * self.delta
        flux_term = self.cfg.lambda_flux * self.flux_proj
        self.decoherence_term = self.cfg.lambda_decoherence * self.Gamma_D * self.u

        self.u = self.u + delta_term + flux_term - self.decoherence_term

        self.history_buffer[1:] = self.history_buffer[:-1]
        self.history_buffer[0] = self.u.copy()

        self.compute_observables_for_current_state(self.u)
        self._capture_frame(frame_index=step_index + 1)
        self._capture_summary(step_index=step_index)

    def run(self) -> Dict[str, Any]:
        self.u_frames = []
        self.summaries = []

        # frame[0]: blank stage
        self.initialize_quiescent_frame()

        # frame[1]: seeded state == step 0 of the 2D-style scenario
        self.inject_seeded_state()

        # frame[2:] evolved states (subject to capture_every for stored frames)
        for step_index in range(1, self.cfg.steps + 1):
            self.step(step_index)

        return {
            "u": self.u.copy(),
            "rho": self.rho.copy(),
            "phi": self.phi.copy(),
            "Gamma": self.Gamma.copy(),
            "R": self.R.copy(),
            "Gamma_D": self.Gamma_D.copy(),
            "Ix": self.Ix.copy(),
            "Iy": self.Iy.copy(),
            "Iz": self.Iz.copy(),
            "summaries": list(self.summaries),
            "u_frames": list(self.u_frames),
        }

    def final_summary(self) -> dict[str, Any]:
        summary = dict(self.summaries[-1]) if self.summaries else self.compute_summary(step_index=-1)
        summary.update(
            {
                "nz": self.cfg.nz,
                "ny": self.cfg.ny,
                "nx": self.cfg.nx,
                "steps": self.cfg.steps,
                "w": self.cfg.w,
                "alpha": self.cfg.alpha,
                "rho_c": self.cfg.rho_c,
                "u0": self.cfg.u0,
                "seed_sign": self.cfg.seed_sign,
                "lambda_delta": self.cfg.lambda_delta,
                "lambda_flux": self.cfg.lambda_flux,
                "lambda_decoherence": self.cfg.lambda_decoherence,
                "operator_mode": "matched",
            }
        )
        return summary

    def validate_basic_state(self) -> dict[str, Any]:
        return {
            "has_nan_u": bool(np.isnan(self.u).any()),
            "has_inf_u": bool(np.isinf(self.u).any()),
            "has_nan_R": bool(np.isnan(self.R).any()),
            "has_inf_R": bool(np.isinf(self.R).any()),
            "max_abs_u": float(np.max(np.abs(self.u))),
            "max_R_proxy": float(np.max(self.R)),
            "min_R_proxy": float(np.min(self.R)),
        }


## Signed 3D Rendering For `u`

Rendering is based on the signed scalar field `u` with a global symmetric colormap. Point selection uses `|u|` with an active-cell filter plus a hard cap on the number of rendered points, while colors remain signed so positive and negative structure stay visually honest.

In [ ]:
def frame_to_point_cloud(
    frame: np.ndarray,
    *,
    max_points: int,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Convert one signed scalar 3D frame into a sparse point cloud.

    Display policy:
    - show all active cells if their count is <= max_points
    - otherwise keep the strongest max_points by |u|

    This preserves signed symmetry while avoiding disappearance of valid
    structure from percentile-based point filtering.
    """
    nz, ny, nx = frame.shape
    cz, cy, cx = nz // 2, ny // 2, nx // 2

    abs_u = np.abs(frame)
    active = abs_u > 1.0e-15

    if not np.any(active):
        return np.empty((0, 3), dtype=np.float64), np.empty((0,), dtype=np.float64)

    z_idx, y_idx, x_idx = np.nonzero(active)
    values = frame[z_idx, y_idx, x_idx]

    if values.size > max_points:
        keep = np.argpartition(np.abs(values), -max_points)[-max_points:]
        z_idx = z_idx[keep]
        y_idx = y_idx[keep]
        x_idx = x_idx[keep]
        values = values[keep]

    xyz = np.column_stack(
        [
            x_idx - cx,
            y_idx - cy,
            z_idx - cz,
        ]
    ).astype(np.float64)

    return xyz, values


def signed_display_limit(frames: list[np.ndarray], percentile: float) -> float:
    """
    One global symmetric display limit for signed u across all stored frames.
    """
    magnitudes = [
        np.abs(frame[np.abs(frame) > 1.0e-15])
        for frame in frames
        if np.any(np.abs(frame) > 1.0e-15)
    ]

    if not magnitudes:
        return 1.0

    all_active = np.concatenate(magnitudes)
    vmax = float(np.percentile(all_active, percentile))
    return max(vmax, 1.0e-12)


def preview_frame_indices(num_frames: int) -> list[int]:
    """
    Return a compact set of preview frames: first, seeded, midpoint, final.
    """
    if num_frames <= 0:
        return []
    raw = [0, min(1, num_frames - 1), num_frames // 2, num_frames - 1]
    return list(dict.fromkeys(raw))


def _quartile_ticks(lo: float, hi: float) -> np.ndarray:
    """
    Return integer-rounded ticks at 0%, 25%, 50%, 75%, 100%.
    """
    return np.unique(np.round(np.linspace(lo, hi, 5)).astype(int))


def _style_clean_3d_axes(ax, xlim, ylim, zlim) -> None:
    """
    Remove pane/background clutter, keep only x-axis quartile ticks.
    """
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_zlim(*zlim)

    ax.set_xticks(_quartile_ticks(*xlim))
    ax.set_yticks([])
    ax.set_zticks([])

    ax.grid(False)

    for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
        axis.pane.fill = False
        axis.pane.set_facecolor((1.0, 1.0, 1.0, 0.0))
        axis.pane.set_edgecolor((1.0, 1.0, 1.0, 0.0))

    ax.xaxis._axinfo["grid"]["linewidth"] = 0.0
    ax.yaxis._axinfo["grid"]["linewidth"] = 0.0
    ax.zaxis._axinfo["grid"]["linewidth"] = 0.0


def render_true3d_frame(
    frame: np.ndarray,
    *,
    max_points: int = 7000,
    cmap: str = "coolwarm",
    display_vmax: float | None = None,
    title: str = "3D U-Field",
    elev: float = 24.0,
    azim: float = -60.0,
):
    """
    Render one sparse signed 3D point cloud with a symmetric colormap for u.
    """
    xyz, values = frame_to_point_cloud(frame, max_points=max_points)

    if display_vmax is None:
        local = np.abs(frame[np.abs(frame) > 1.0e-15])
        display_vmax = float(np.max(local)) if local.size else 1.0

    display_vmax = max(display_vmax, 1.0e-12)
    norm = colors.TwoSlopeNorm(vmin=-display_vmax, vcenter=0.0, vmax=display_vmax)

    nz, ny, nx = frame.shape
    cz, cy, cx = nz // 2, ny // 2, nx // 2

    xlim = (-cx, nx - cx - 1)
    ylim = (-cy, ny - cy - 1)
    zlim = (-cz, nz - cz - 1)

    fig = plt.figure(figsize=(8, 8))
    ax = fig.add_subplot(111, projection="3d")

    if values.size:
        ax.scatter(
            xyz[:, 0], xyz[:, 1], xyz[:, 2],
            c=values,
            cmap=cmap,
            norm=norm,
            s=8,
            alpha=0.85,
            linewidths=0.0,
        )

    _style_clean_3d_axes(ax, xlim, ylim, zlim)

    ax.set_xlabel("x")
    ax.set_ylabel("")
    ax.set_zlabel("")
    ax.set_title(title)
    ax.view_init(elev=elev, azim=azim)
    ax.set_box_aspect((xlim[1] - xlim[0], ylim[1] - ylim[0], zlim[1] - zlim[0]))

    return fig, ax


def animate_true3d_u_frames(
    frames: list[np.ndarray],
    *,
    seed_sign: int = 1,
    max_points: int = 7000,
    cmap: str = "coolwarm",
    display_percentile: float = 99.5,
    interval_ms: int = 80,
    title: str = "3D U-Field",
    elev: float = 24.0,
    azim: float = -60.0,
):
    """
    Build a true-3D signed point-cloud animation of the scalar u field.
    """
    vmax = signed_display_limit(frames, percentile=display_percentile)
    norm = colors.TwoSlopeNorm(vmin=-vmax, vcenter=0.0, vmax=vmax)

    nz, ny, nx = frames[0].shape
    cz, cy, cx = nz // 2, ny // 2, nx // 2

    xlim = (-cx, nx - cx - 1)
    ylim = (-cy, ny - cy - 1)
    zlim = (-cz, nz - cz - 1)

    fig = plt.figure(figsize=(8, 8))
    ax = fig.add_subplot(111, projection="3d")

    _style_clean_3d_axes(ax, xlim, ylim, zlim)

    ax.set_xlabel("x")
    ax.set_ylabel("")
    ax.set_zlabel("")
    ax.view_init(elev=elev, azim=azim)
    ax.set_box_aspect((xlim[1] - xlim[0], ylim[1] - ylim[0], zlim[1] - zlim[0]))

    scat = ax.scatter([], [], [], c=[], cmap=cmap, norm=norm, s=8, alpha=0.85, linewidths=0.0)

    def update(i: int):
        xyz, values = frame_to_point_cloud(frames[i], max_points=max_points)

        frame = frames[i]
        u_max = float(np.max(frame))
        u_min = float(np.min(frame))
        pos_fraction = float(np.count_nonzero(frame > 0) / frame.size)
        neg_fraction = float(np.count_nonzero(frame < 0) / frame.size)
        seed_label = "+" if seed_sign > 0 else "-"

        if xyz.size == 0:
            scat._offsets3d = ([], [], [])
            scat.set_array(np.array([]))
        else:
            scat._offsets3d = (xyz[:, 0], xyz[:, 1], xyz[:, 2])
            scat.set_array(values)

        ax.set_title(
            f"{title} | Frame {i} | Seed: {seed_label}\n"
            f"u_max/min: {u_max:.2f}, {u_min:.2f} | POS/NEG: {pos_fraction:.2f}, {neg_fraction:.2f}"
        )
        return (scat,)

    ani = animation.FuncAnimation(fig, update, frames=len(frames), interval=interval_ms, blit=False)
    plt.close(fig)
    return ani


def export_true3d_video_mp4(
    frames: list[np.ndarray],
    *,
    output_mp4: str,
    seed_sign: int = 1,
    max_points: int = 7000,
    cmap: str = "coolwarm",
    display_percentile: float = 99.5,
    fps: int = 12,
    interval_ms: int = 80,
    title: str = "3D U-Field",
    elev: float = 24.0,
    azim: float = -60.0,
) -> str:
    """
    Export the signed 3D scalar-u animation to MP4 with one global symmetric color scale.
    """
    ani = animate_true3d_u_frames(
        frames,
        seed_sign=seed_sign,
        max_points=max_points,
        cmap=cmap,
        display_percentile=display_percentile,
        interval_ms=interval_ms,
        title=title,
        elev=elev,
        azim=azim,
    )

    writer = animation.FFMpegWriter(fps=fps, bitrate=2400)
    ani.save(output_mp4, writer=writer)
    return str(Path(output_mp4).resolve())


## **Run Compact 3D Simulation**





In [ ]:

cfg = ScalarConfig(
    nz=75,                            # (15, 25, 33, 51, 75)
    ny=75,                            # (15, 25, 33, 51, 75)
    nx=75,                            # (15, 25, 33, 51, 75)
    steps=500,                        # (120, 250, 500)
    w=6,                              # (6)
    alpha=10.0,                       # (10.0)
    rho_c=0.5,                        # (0.5)
    u0=1.0e-3,                        # (1.0e-3)
    seed_sign=1,                      # (1 or -1)
    lambda_delta=1.00,                # (1.00)
    lambda_flux=0.50,                 # (0.50)
    lambda_decoherence=0.25,          # (0.25)
    max_points=20000,                 # (7000, 20000)
    cmap="coolwarm",                  # ("coolwarm")
    export_display_percentile=99.5,   # (99.5)
    fps=12,                           # (12)
    interval_ms=80,                   # (80)
    output_mp4="3d_u_field.mp4",
    title="3D U-Field",
)

sim = Scalar3DSim(cfg)
results = sim.run()

print("\nFinal validation:")
print(sim.validate_basic_state())

print("\nFinal summary:")
for k, v in sim.final_summary().items():
    print(f"{k}: {v}")



## Preview Key Frames

In [ ]:
display_vmax = signed_display_limit(results["u_frames"], percentile=cfg.export_display_percentile)

for frame_idx in preview_frame_indices(len(results["u_frames"])):
    if frame_idx == 0:
        label = "blank stage"
    elif frame_idx == 1:
        label = "seeded state"
    elif frame_idx == len(results["u_frames"]) - 1:
        label = "final state"
    else:
        label = "mid-run preview"

    fig, ax = render_true3d_frame(
        results["u_frames"][frame_idx],
        max_points=cfg.max_points,
        cmap=cfg.cmap,
        display_vmax=display_vmax,
        title=f"Frame {frame_idx} | {label}",
    )
    plt.show()


## MP4 export

In [ ]:
mp4_path = export_true3d_video_mp4(
    results["u_frames"],
    output_mp4=cfg.output_mp4,
    seed_sign=cfg.seed_sign,
    max_points=cfg.max_points,
    cmap=cfg.cmap,
    display_percentile=cfg.export_display_percentile,
    fps=cfg.fps,
    interval_ms=cfg.interval_ms,
    title=cfg.title,
)

print(f"Saved MP4: {mp4_path}")

try:
    from google.colab import files  # type: ignore
    files.download(mp4_path)
except Exception:
    print("Automatic download is only available in Colab. The MP4 is saved locally.")
